# AIRF SLM tool calling — Colab training (target ≥ 93% dev)

Run on **Colab GPU (T4)**. Secrets:
- `HF_TOKEN` — Hugging Face token (read access)
- optional `GH_TOKEN` — push `submission.zip`

**Gated models:** FunctionGemma and Gemma need **Agree and access** on the model page. A 403 means the token is missing or the licence is not accepted.

This notebook tries, in order:
1. `google/functiongemma-270m-it` (best for tools, gated)
2. `google/gemma-3-270m-it` (official starter, gated)
3. `Qwen/Qwen2.5-1.5B-Instruct` (open, ≤ 4B)

Then LoRA, 2 epochs, full train, `submission.zip`.


In [ ]:
import importlib.util, subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao", "peft", "accelerate", "transformers", "huggingface_hub"])
print("ok")


In [ ]:
import json, math, os, random, re, time, urllib.request, zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
print("device", DEVICE, DTYPE)

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if token:
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF login ok")
else:
    print("WARNING: no HF_TOKEN — gated Gemma/FunctionGemma will 403. Using open models only.")


In [ ]:
REPO = "https://raw.githubusercontent.com/aims-ai-research-foundations/airf-slm-tool-calling-hackathon/main/data"
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
for name in ("tools.json", "train.jsonl", "dev.jsonl"):
    urllib.request.urlretrieve(f"{REPO}/{name}", DATA_DIR / name)

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

def answer_of(row):
    return row["tool_calls"] if "tool_calls" in row else {"tool": row["tool"], "parameters": row["parameters"]}

def calls_of(answer):
    return answer if isinstance(answer, list) else [answer]

tools = {t["name"]: t for t in json.loads((DATA_DIR / "tools.json").read_text())}
train = [dict(row, answer=answer_of(row)) for row in read_jsonl(DATA_DIR / "train.jsonl")]
dev = read_jsonl(DATA_DIR / "dev.jsonl")
print(len(tools), "tools", len(train), "train", len(dev), "dev")


In [ ]:
CANDIDATES = [
    os.environ.get("MODEL_NAME") or "google/functiongemma-270m-it",
    "google/gemma-3-270m-it",
    "Qwen/Qwen2.5-1.5B-Instruct",
]
seen = set()
CANDIDATES = [m for m in CANDIDATES if not (m in seen or seen.add(m))]

EPOCHS = 2
LORA_R = 32
LR = 1.5e-4
INSTRUCTION = (
    "Reply with a JSON list of the tool calls for the request, with null for values not given.\n"
    "Every parameter of the chosen tool must be present. Do not invent values.\nTools:\n"
)

MODEL_NAME = None
last_err = None
for cand in CANDIDATES:
    try:
        print("trying", cand)
        tokenizer = AutoTokenizer.from_pretrained(cand)
        MODEL_NAME = cand
        print("using", MODEL_NAME)
        break
    except Exception as e:
        last_err = e
        print("skip", cand, "->", type(e).__name__, str(e)[:240])

if MODEL_NAME is None:
    raise RuntimeError(
        "No model could be loaded. Accept licences at "
        "https://huggingface.co/google/gemma-3-270m-it and "
        "https://huggingface.co/google/functiongemma-270m-it "
        "then put HF_TOKEN in Colab Secrets. Last error: " + repr(last_err)
    )

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(example):
    signatures = [f"{name}({', '.join(f'{p}: {t}' for p, t in tools[name]['parameters'].items())})" for name in example["tools"]]
    messages = [{"role": "system", "content": INSTRUCTION + "\n".join(signatures)}, {"role": "user", "content": example["request"]}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def target_text(answer):
    return json.dumps(calls_of(answer), ensure_ascii=False) + "<end_of_turn>"

by_category = defaultdict(list)
for ex in train:
    by_category[ex["meta"]["category"]].append(ex)
validation = [ex for group in by_category.values() for ex in random.Random(SEED).sample(group, 100)]
held = {ex["id"] for ex in validation}
train_pool = [ex for ex in train if ex["id"] not in held]
print(MODEL_NAME, "train_pool", len(train_pool), "val", len(validation))


In [ ]:
def normalise(value):
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    if isinstance(value, str):
        return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", value.lower())).strip()
    return value

def canonical(answer):
    try:
        return tuple((c["tool"], tuple(sorted((k, normalise(v)) for k, v in c["parameters"].items() if v is not None))) for c in calls_of(answer))
    except (TypeError, KeyError, AttributeError):
        return None

def score(predictions):
    hits = defaultdict(list)
    for ex in validation:
        hits[ex["meta"]["category"]].append(canonical(predictions.get(ex["id"])) == canonical(ex["answer"]))
    return {"overall": float(np.mean(sum(hits.values(), [])))} | {k: float(np.mean(hits[k])) for k in ("single", "missing_info", "multiple")}

def parse(text):
    text = text.split("<end_of_turn>")[0]
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    try:
        value = json.JSONDecoder().raw_decode(text, min(starts))[0] if starts else None
    except json.JSONDecodeError:
        return None
    return value[0] if isinstance(value, list) and len(value) == 1 else value

@torch.no_grad()
def predict(model, examples, batch_size=16):
    model.eval()
    order = sorted(examples, key=lambda ex: len(build_prompt(ex)))
    stop = [tokenizer.eos_token_id]
    eot = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if isinstance(eot, int) and eot >= 0:
        stop.append(eot)
    out = {}
    for i in range(0, len(order), batch_size):
        batch = order[i:i+batch_size]
        inputs = tokenizer([build_prompt(ex) for ex in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        tokens = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=stop, pad_token_id=tokenizer.pad_token_id)
        for ex, seq in zip(batch, tokens[:, inputs["input_ids"].shape[1]:]):
            out[ex["id"]] = parse(tokenizer.decode(seq, skip_special_tokens=False))
        if i % 128 == 0:
            print("pred", i, "/", len(order))
    return out


In [ ]:
def fine_tune(examples, epochs=2, lr=LR, batch_tokens=4096, lora_r=LORA_R):
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation="eager").to(DEVICE)
    model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", r=lora_r, lora_alpha=lora_r*2, target_modules="all-linear"))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    rows = []
    for ex in examples:
        prompt = tokenizer(build_prompt(ex), add_special_tokens=False)["input_ids"]
        answer = tokenizer(target_text(ex["answer"]), add_special_tokens=False)["input_ids"]
        rows.append((prompt + answer, [-100]*len(prompt) + answer))
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    start = time.time()
    base = model.get_base_model()
    for ep in range(epochs):
        random.Random(SEED+ep).shuffle(rows)
        packed = sorted(rows, key=lambda r: len(r[0]) // 32)
        batches, batch = [], []
        for row in packed:
            if batch and (len(batch)+1)*len(row[0]) > batch_tokens:
                batches.append(batch); batch = []
            batch.append(row)
        batches.append(batch)
        random.Random(SEED+ep).shuffle(batches)
        nsteps = max(1, len(batches))
        schedule = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lambda s, n=nsteps: min(1, (s+1)/20) * 0.5 * (1 + math.cos(math.pi * s / n))
        )
        model.train()
        for bi, batch in enumerate(batches):
            width = max(len(ids) for ids, _ in batch)
            ids = torch.tensor([[tokenizer.pad_token_id]*(width-len(i))+i for i,_ in batch], device=DEVICE)
            labels = torch.tensor([[-100]*(width-len(l))+l for _,l in batch], device=DEVICE)
            mask = (ids != tokenizer.pad_token_id).long()
            hidden = base.model(input_ids=ids, attention_mask=mask, position_ids=(mask.cumsum(1)-1).clamp(min=0)).last_hidden_state
            targets, keep = labels[:,1:], labels[:,1:] != -100
            logits = base.lm_head(hidden[:,:-1][keep]).float()
            loss = torch.nn.functional.cross_entropy(logits, targets[keep])
            if math.isfinite(loss.item()):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            schedule.step(); optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())
            if bi % 50 == 0:
                print(f"epoch {ep+1}/{epochs} step {bi}/{len(batches)} loss {loss.item():.4f}")
    print(f"done {len(examples)} x {epochs} ep in {(time.time()-start)/60:.1f} min")
    return model, losses


In [ ]:
model, losses = fine_tune(train_pool, epochs=EPOCHS)
val_score = score(predict(model, validation))
print("validation", val_score)
if val_score["overall"] < 0.90:
    print("Val < 90%. If you used Qwen because of 403, accept Gemma/FunctionGemma and rerun, or set MODEL_NAME=google/gemma-3-1b-it")


In [ ]:
adapter_dir = Path("adapter")
adapter_dir.mkdir(exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved", adapter_dir.resolve())

predictions = predict(model, dev)
empty = {"tool": "", "parameters": {}}
payload = {k: (v if v is not None else empty) for k, v in predictions.items()}
Path("submission.json").write_text(json.dumps(payload, ensure_ascii=False, indent=1))
Path("predictions.json").write_text(json.dumps(payload, ensure_ascii=False))
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("predictions.json")
    z.write("submission.json")
print(len(payload), "-> submission.zip  (upload this on Codabench)")


In [ ]:
GH_REPO = os.environ.get("GH_REPO", "https://github.com/maick-code/airf-slm-tool-calling-hackathon.git")
BRANCH = os.environ.get("GH_BRANCH", "arena/7d628867-airf-slm-tool-calling-hackatho")
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = os.environ.get("GH_TOKEN")
if token:
    import subprocess
    auth = GH_REPO.replace("https://", f"https://x-access-token:{token}@")
    subprocess.check_call(["git", "config", "user.email", "colab@local"])
    subprocess.check_call(["git", "config", "user.name", "colab"])
    subprocess.call(["git", "init"])
    subprocess.call(["git", "remote", "add", "origin", auth])
    subprocess.call(["git", "fetch", "origin", BRANCH])
    subprocess.call(["git", "checkout", "-B", BRANCH])
    subprocess.check_call(["git", "add", "submission.json", "submission.zip"])
    subprocess.call(["git", "commit", "-m", "Colab submission"])
    subprocess.check_call(["git", "push", "origin", BRANCH])
    print("pushed to", BRANCH)
else:
    print("No GH_TOKEN — download submission.zip from Colab files and upload to Codabench.")
